In [2]:
# CELL 1: Setup and Configuration
import pandas as pd
from datasets import load_dataset
from google.colab import userdata
import os
from concurrent.futures import ThreadPoolExecutor
from google.colab import drive
HF_TOKEN = userdata.get('HF_TOKEN')
selected_subs = ['philosophy', 'todayilearned', 'technology']

target_counts = {
    'philosophy': 3305,
    'todayilearned': 2445,
    'technology': 1876
}

drive.mount('/content/drive')

# 2. Create a dedicated cache folder in your Drive


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
drive_cache_dir = '/content/drive/MyDrive/Colab Notebooks/data/huggingface_cache'
os.makedirs(drive_cache_dir, exist_ok=True)

In [3]:
# CELL 2: Submissions Loading and Sampling
print("Loading submissions from REDDIT_submissions...")
submissions_dfs = []

for subreddit in selected_subs:
    print(f"=== Loading submissions from r/{subreddit} ===")
    dataset = load_dataset(
        "HuggingFaceGECLM/REDDIT_submissions",
        split=subreddit,
        token=HF_TOKEN,
        cache_dir=drive_cache_dir
    )
    df_sub = dataset.to_pandas()
    df_sub['subreddit'] = subreddit
    submissions_dfs.append(df_sub)

all_submissions = pd.concat(submissions_dfs, ignore_index=True)

# Convert Unix timestamp to Datetime
all_submissions['created_utc_dt'] = pd.to_datetime(all_submissions['created_utc'], unit='s')

# Determine the most recent 12-day window globally across the loaded dataset
max_date = all_submissions['created_utc_dt'].max()
start_date = max_date - pd.Timedelta(days=12)

# print(f"\nFiltering temporal window: {start_date} to {max_date}")
# recent_submissions = all_submissions[
#     (all_submissions['created_utc_dt'] >= start_date) &
#     (all_submissions['created_utc_dt'] <= max_date)
# ]

# print("Performing volume-matched stratified random sampling...")
# sampled_dfs = []
# for subreddit, target in target_counts.items():
#     sub_df = recent_submissions[recent_submissions['subreddit'] == subreddit]
#     sampled = sub_df.sample(n=target, random_state=42)
#     sampled_dfs.append(sampled)

# sampled_submissions = pd.concat(sampled_dfs, ignore_index=True)
# print(f"Total sampled submissions: {len(sampled_submissions):,}")

Loading submissions from REDDIT_submissions...
=== Loading submissions from r/philosophy ===


README.md: 0.00B [00:00, ?B/s]

KeyboardInterrupt: 

In [ ]:
# all_submissions = pd.concat(submissions_dfs, ignore_index=True)
# start_date = pd.to_datetime('2023-01-01')
# end_date = pd.to_datetime('2023-01-31 23:59:59')
# # Convert Unix timestamp to Datetime
# all_submissions['created_utc_dt'] = pd.to_datetime(all_submissions['created_utc'], unit='s')

# print(f"\nFiltering temporal window: {start_date} to {end_date}")
# jan_2023_submissions = all_submissions[
#     (all_submissions['created_utc_dt'] >= start_date) &
#     (all_submissions['created_utc_dt'] <= end_date)
# ]

# print("Performing volume-matched stratified random sampling...")


In [ ]:
# import shutil
# import os

# # Define the cache path
# cache_path = os.path.expanduser("~/.cache/huggingface")

# # Remove the entire cache directory
# if os.path.exists(cache_path):
#     shutil.rmtree(cache_path)
#     print(f"Cleared Hugging Face cache at: {cache_path}")
# else:
#     print("Hugging Face cache is already empty.")

In [ ]:
# all_submissions  = pd.read_csv('all_submissions.csv')
# all_submissions

In [5]:
DESTINATION_DIR = '/content/drive/MyDrive/Colab Notebooks/data/'

In [ ]:
all_submissions.to_csv(f'{DESTINATION_DIR}/all_submissions.csv', index=False)

In [5]:
all_submissions = pd.read_csv(f'{DESTINATION_DIR}/all_submissions.csv')

/tmp/ipykernel_36971/2905392034.py:1: DtypeWarning: Columns (0,1,3,7,10,12,14,15,17,18,19,20,21,22,24,25,26,28,29,30,34,36,37,39,41,44,45,46,47,48,49,51,53,54,56,58,61,63) have mixed types. Specify dtype option on import or set low_memory=False.
  all_submissions = pd.read_csv(f'{DESTINATION_DIR}/all_submissions.csv')


In [6]:
all_submissions.columns

Index(['allow_live_comments', 'archived', 'author', 'author_fullname',
       'banned_by', 'category', 'content_categories', 'contest_mode',
       'created_utc', 'discussion_type', 'distinguished', 'domain', 'edited',
       'gilded', 'hidden', 'hide_score', 'id', 'is_created_from_ads_ui',
       'is_crosspostable', 'is_meta', 'is_original_content',
       'is_reddit_media_domain', 'is_robot_indexable', 'is_self', 'is_video',
       'locked', 'media', 'media_embed', 'media_only', 'name', 'no_follow',
       'num_comments', 'num_crossposts', 'over_18', 'parent_whitelist_status',
       'permalink', 'pinned', 'post_hint', 'pwls', 'quarantine', 'removed_by',
       'removed_by_category', 'retrieved_on', 'score', 'secure_media',
       'secure_media_embed', 'selftext', 'send_replies', 'spoiler', 'stickied',
       'subreddit_id', 'subreddit_name_prefixed', 'subreddit_subscribers',
       'subreddit_type', 'suggested_sort', 'title', 'top_awarded_type',
       'total_awards_received', 'trea

In [7]:
invalid_markers = ['[deleted]', '[removed]']

# ==========================================
# OPTION A: KEEP ONLY THE CLEAN ROWS (Recommended)
# ==========================================
# 1. Drop rows where selftext is NaN
clean_submissions = all_submissions.dropna(subset=['selftext']).copy()

# 2. Filter out rows where selftext is exactly '[deleted]' or '[removed]'
clean_submissions = clean_submissions[~clean_submissions['selftext'].isin(invalid_markers)]

print(f"Cleaned submissions count: {len(clean_submissions):,}")
print(f"Removed {len(all_submissions) - len(clean_submissions):,} invalid rows.\n")
# clean_submissions.to_csv('clean_submissions.csv', index=False)

# Overwrite the original variable with the clean data
all_submissions = clean_submissions

Cleaned submissions count: 49,838
Removed 4,429,375 invalid rows.



In [8]:
clean_submissions.to_csv(f'{DESTINATION_DIR}/clean_submissions.csv', index=False)

In [ ]:
# start_date = pd.to_datetime('2023-01-01')
# end_date = pd.to_datetime('2023-01-31 23:59:59')
# all_submissions['created_utc_dt'] = pd.to_datetime(all_submissions['created_utc'], unit='s')
# jan_2023_submissions = all_submissions[
#     (all_submissions['created_utc_dt'] >= start_date) &
#     (all_submissions['created_utc_dt'] <= end_date)
# ]

In [9]:
print("\nExtracting the highest-commented posts for each subreddit...")
sampled_dfs = []
for subreddit, target in target_counts.items():
    sub_df = all_submissions[all_submissions['subreddit'] == subreddit]
    available_count = len(sub_df)

    sample_size = min(target, available_count)

    # NEW: Grab the top 'sample_size' posts with the highest number of comments
    sampled = sub_df.nlargest(sample_size, 'num_comments')

    # Print the minimum threshold we hit for this sub so you know the engagement floor
    min_comments = sampled['num_comments'].min()
    print(f"r/{subreddit} -> Target: {target:,} | Sampling: {sample_size:,} | Lowest comment count in sample: {min_comments}")

    sampled_dfs.append(sampled)


Extracting the highest-commented posts for each subreddit...
r/philosophy -> Target: 3,305 | Sampling: 3,305 | Lowest comment count in sample: 45
r/todayilearned -> Target: 2,445 | Sampling: 2,445 | Lowest comment count in sample: 8
r/technology -> Target: 1,876 | Sampling: 1,876 | Lowest comment count in sample: 20


In [ ]:
# sampled_dfs = []
# for subreddit, target in target_counts.items():
#     sub_df = all_submissions[all_submissions['subreddit'] == subreddit]
#     available_count = len(sub_df)

#     # Safely take the target amount, or all available rows if there's a deficit
#     sample_size = min(target, available_count)

#     print(f"r/{subreddit} -> Target: {target:,} | Available: {available_count:,} | Sampling: {sample_size:,}")

#     if sample_size < target:
#         print(f"  * Shortfall of {target - sample_size:,} posts.")

#     sampled = sub_df.sample(n=sample_size, random_state=42)
#     sampled_dfs.append(sampled)

In [ ]:
# philosophy_df = all_submissions[all_submissions['subreddit'] == 'philosophy']
# philosophy_available = len(philosophy_df)
# philosophy_shortfall = target_counts['philosophy'] - philosophy_available

# if philosophy_shortfall > 0:
#     print(f"\nDetected a shortfall of {philosophy_shortfall:,} in r/philosophy.")
#     print("Redistributing deficit evenly to r/todayilearned and r/technology...")
#     target_counts['philosophy'] = philosophy_available

#     # Add 460 to the other two categories
#     target_counts['todayilearned'] += (philosophy_shortfall // 2)
#     target_counts['technology'] += (philosophy_shortfall // 2)

# print("\nPerforming volume-matched stratified random sampling with adjusted targets...")
# sampled_dfs = []

# for subreddit, target in target_counts.items():
#     sub_df = all_submissions[all_submissions['subreddit'] == subreddit]
#     available_count = len(sub_df)

#     sample_size = min(target, available_count)
#     print(f"r/{subreddit} -> Adjusted Target: {target:,} | Available: {available_count:,} | Sampling: {sample_size:,}")

#     sampled = sub_df.sample(n=sample_size, random_state=42)
#     sampled_dfs.append(sampled)

# sampled_submissions = pd.concat(sampled_dfs, ignore_index=True)
# print(f"\nTotal sampled submissions for Jan 2023: {len(sampled_submissions):,}")

In [10]:
sampled_submissions = pd.concat(sampled_dfs, ignore_index=True)

In [12]:
sampled_submissions["selftext"]

,selftext
0,"EDIT: Wow, I didn't realize the hornet's nest ..."
1,Philosophy should be just as important as math...
2,Hope this sub is appropriate. Any simplificati...
3,"The way I see it, life is inherently worthless..."
4,Would you take a bet that you only had a 10% c...
...,...
7621,As we all know the current circlejerk about th...
7622,Dumb question I'm sure but I feel like I only ...
7623,I have been posting this in comments but decid...
7624,Posting this in this sub because it is where I...


In [6]:
sampled_submissions = pd.read_csv(f'{DESTINATION_DIR}/sampled_submissions.csv')

In [13]:
# sampled_submissions.to_csv(f'{DESTINATION_DIR}/sampled_submissions.csv', index=False)

In [ ]:
len(selected_subs)

In [7]:
import pandas as pd
from datasets import load_dataset
import gc
import sys

# 1. Validation Check for sampled_submissions
if 'id' not in sampled_submissions.columns or sampled_submissions.empty:
    print("ERROR: sampled_submissions is empty or missing the 'id' column.")
    sys.exit("Stopping execution.")

# 2. THE CHANGE: Create a set of valid link_ids by prefixing the post IDs with 't3_'
valid_link_ids = set('t3_' + sampled_submissions['id'].astype(str))

print(f"\nIdentified {len(valid_link_ids):,} unique post IDs. Starting RAM-optimized extraction for thread comments...")

# Keep the timestamp check to maintain your strict January 2023 temporal window
start_ts = int(pd.to_datetime('2023-01-01').timestamp())
end_ts = int(pd.to_datetime('2023-01-31 23:59:59').timestamp())

comments_dfs = []

# Process STRICTLY sequentially to prevent Colab Pro RAM spiking
for subreddit in selected_subs:
    print(f"\n=== Processing r/{subreddit} ===")

    # Load dataset (Memory-mapped via Arrow)
    dataset = load_dataset(
        "HuggingFaceGECLM/REDDIT_comments",
        split=subreddit,
        token=HF_TOKEN,
        cache_dir=drive_cache_dir
    )

    print("  Filtering comments matching the sampled posts natively in Hugging Face...")

    # THE CHANGE: Filter by link_id instead of author
    def filter_comments(example):
        return (example['link_id'] in valid_link_ids) and (start_ts <= int(example['created_utc']) <= end_ts)

    filtered_dataset = dataset.filter(filter_comments, num_proc=4)

    print("  Converting filtered subset to Pandas...")
    df = filtered_dataset.to_pandas()
    df['subreddit'] = subreddit
    comments_dfs.append(df)

    print(f"  Finished r/{subreddit}: Extracted {len(df):,} thread comments.")

    # Aggressive Garbage Collection
    del dataset
    del filtered_dataset
    gc.collect()

# Combine the pre-filtered dataframes
if comments_dfs:
    post_centric_comments = pd.concat(comments_dfs, ignore_index=True)
    post_centric_comments['created_utc_dt'] = pd.to_datetime(post_centric_comments['created_utc'], unit='s')
    print(f"\n=== SUCCESS ===")
    print(f"Extracted {len(post_centric_comments):,} total comments belonging to the sampled posts.")
else:
    post_centric_comments = pd.DataFrame()
    print("\nNo comments found matching the criteria.")


Identified 7,626 unique post IDs. Starting RAM-optimized extraction for thread comments...

=== Processing r/philosophy ===


Resolving data files:   0%|          | 0/21 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/19 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/27 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

...

Loading dataset shards:   0%|          | 0/22 [00:00<?, ?it/s]

  Filtering comments matching the sampled posts natively in Hugging Face...


Filter (num_proc=4):   0%|          | 0/25404699 [00:00<?, ? examples/s]

  Converting filtered subset to Pandas...
  Finished r/technology: Extracted 10 thread comments.

=== SUCCESS ===
Extracted 494 total comments belonging to the sampled posts.


/tmp/ipykernel_40751/961051899.py:57: FutureWarning: The behavior of 'to_datetime' with 'unit' when parsing strings is deprecated. In a future version, strings will be parsed as datetime strings, matching the behavior without a 'unit'. To retain the old behavior, explicitly cast ints or floats to numeric type before calling to_datetime.
  post_centric_comments['created_utc_dt'] = pd.to_datetime(post_centric_comments['created_utc'], unit='s')


In [8]:
post_centric_comments.to_csv(f'{DESTINATION_DIR}/post_centric_comments.csv', index=False)

In [ ]:
author_recent_comments.to_csv(f'{DESTINATION_DIR}/author_recent_comments2.csv', index=False)

In [9]:
post_centric_comments

,archived,author,author_fullname,body,comment_type,controversiality,created_utc,edited,gilded,id,...,parent_id,permalink,retrieved_on,score,subreddit_id,subreddit_name_prefixed,subreddit_type,total_awards_received,subreddit,created_utc_dt
0,False,Enaiii,t2_3sjjdfph,"Hello! I'm a 'noob' in philosophy, I'm trying ...",None,0,1674358449,False,0,j5d81ze,...,t3_10df9ua,/r/philosophy/comments/10df9ua/rphilosophy_ope...,1676120244,1,t5_2qh5b,r/philosophy,public,0,philosophy,2023-01-22 03:34:09
1,False,Perrr333,t2_13xoyu,Yeah. But it's also important not to stray too...,None,0,1674381060,False,0,j5e62x5,...,t1_j5ck05l,/r/philosophy/comments/10df9ua/rphilosophy_ope...,1676119394,1,t5_2qh5b,r/philosophy,public,0,philosophy,2023-01-22 09:51:00
2,False,DirtyOldPanties,t2_qcs77er3,I disagree.,None,0,1674385137,False,0,j5eawkl,...,t1_j4umi8r,/r/philosophy/comments/10df9ua/rphilosophy_ope...,1676119274,1,t5_2qh5b,r/philosophy,public,0,philosophy,2023-01-22 10:58:57
3,False,el_miguel42,t2_10se7j,I dont... \n\nWhatever I could think up would ...,None,0,1674387277,False,0,j5edij5,...,t1_j5ceqsd,/r/philosophy/comments/10df9ua/rphilosophy_ope...,1676119209,1,t5_2qh5b,r/philosophy,public,0,philosophy,2023-01-22 11:34:37
4,False,CoolGovernment8732,t2_7mphzi9e,"You are right, and that is actually one of the...",None,0,1674402749,False,0,j5f4xnb,...,t1_j4lhz5d,/r/philosophy/comments/10df9ua/rphilosophy_ope...,1676118525,2,t5_2qh5b,r/philosophy,public,0,philosophy,2023-01-22 15:52:29
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
489,False,veritanuda,t2_6l9t3,Need to be more specific about your equipment....,None,0,1675102519,False,0,j6j2vey,...,t1_j6itqo0,/r/technology/comments/10omwh7/rtechnology_biw...,1676083789,2,t5_2qh16,r/technology,public,0,technology,2023-01-30 18:15:19
490,False,TryHot9052,t2_eipnai7j,Thank you!,None,0,1675102737,False,0,j6j3gj0,...,t1_j6j2vey,/r/technology/comments/10omwh7/rtechnology_biw...,1676083774,1,t5_2qh16,r/technology,public,0,technology,2023-01-30 18:18:57
491,False,Deathcon2004,t2_i2oz4atp,I had one of those small iPhones (iPhone SE I ...,None,0,1675206113,False,0,j6pbk1x,...,t3_10omwh7,/r/technology/comments/10omwh7/rtechnology_biw...,1676078395,1,t5_2qh16,r/technology,public,0,technology,2023-01-31 23:01:53
492,False,[deleted],None,[removed],None,0,1675209091,False,0,j6piv5h,...,t3_10omwh7,/r/technology/comments/10omwh7/rtechnology_biw...,1676078222,1,t5_2qh16,r/technology,public,0,technology,2023-01-31 23:51:31


In [ ]:
author_recent_comments.columns

In [ ]:
print("\nFormatting Unified Interaction Stream...")

# Standardize text columns (Submissions have title/selftext, Comments have body)
sampled_submissions = sampled_submissions.copy()
sampled_submissions['interaction_type'] = 'post'
sampled_submissions['text'] = sampled_submissions['title'].fillna('') + ' ' + sampled_submissions['selftext'].fillna('')

author_recent_comments['interaction_type'] = 'comment'
author_recent_comments['text'] = author_recent_comments['body'].fillna('')

# Select common columns for the long format
cols_to_keep = ['author', 'created_utc_dt', 'subreddit', 'interaction_type',
                'text', 'score' ,'id']

# Safely extract the columns that exist in the respective dataframes
sub_cols = [c for c in cols_to_keep if c in sampled_submissions.columns]
com_cols = [c for c in cols_to_keep if c in author_recent_comments.columns]

unified_stream = pd.concat([
    sampled_submissions[sub_cols],
    author_recent_comments[com_cols]
], ignore_index=True)

# Sort chronologically for each author to recreate their exact interaction sequence
unified_stream = unified_stream.sort_values(by=['author', 'created_utc_dt']).reset_index(drop=True)
unified_stream["label"] = 1
unified_stream.to_csv(f'{DESTINATION_DIR}/unified_stream2.csv', index=False)
print("\n=== FINAL UNIFIED STREAM ===")
print(unified_stream.head())
print(f"\nFinal Dataset Shape: {unified_stream.shape}")

In [ ]:
author_recent_comments['clean_link_id'] = author_recent_comments['link_id'].str.replace('t3_', '')

# parent_id could be either t1_ or t3_, so we strip both possibilities
author_recent_comments['clean_parent_id'] = author_recent_comments['parent_id'].str.replace('t1_', '').str.replace('t3_', '')

# 2. Classify the depth of the interaction
author_recent_comments['is_direct_reply_to_post'] = author_recent_comments['parent_id'].str.startswith('t3_')
author_recent_comments['is_reply_to_comment'] = author_recent_comments['parent_id'].str.startswith('t1_')

# 3. Match the comments to their Root Post (Submission) context
# We merge the comment's clean_link_id with the submission's id
# (Assuming 'all_submissions' is still in your memory from the earlier steps)
comments_with_context = pd.merge(
    author_recent_comments,
    sampled_submissions[['id', 'title', 'selftext', 'author']],
    left_on='clean_link_id',
    right_on='id',
    how='left',
    suffixes=('_comment', '_root_post')
)

# Rename columns for clarity
comments_with_context = comments_with_context.rename(columns={
    'title': 'root_post_title',
    'selftext': 'root_post_text',
    'author_root_post': 'root_post_author',
    'author_comment': 'author'
})

# Drop the redundant 'id_root_post' column that came from the merge
if 'id_root_post' in comments_with_context.columns:
    comments_with_context = comments_with_context.drop(columns=['id_root_post'])

print(f"Successfully mapped {len(comments_with_context):,} comments to their root posts.")
print("\n=== Sample of Mapped Data ===")
print(comments_with_context[['author', 'body', 'is_reply_to_comment', 'root_post_title']].head())

In [11]:
sampled_submissions = sampled_submissions.copy()
sampled_submissions['interaction_type'] = 'submission'

# Combine title and text for a complete view of the prompt
sampled_submissions['content'] = sampled_submissions['title'].fillna('') + '\n\n' + sampled_submissions['selftext'].fillna('')

# Root posts have no parent, and they act as their own root ID
sampled_submissions['clean_parent_id'] = None
sampled_submissions['clean_root_id'] = sampled_submissions['id']


# ==========================================
# 2. Format Comments
# ==========================================
post_centric_comments = post_centric_comments.copy()
post_centric_comments['interaction_type'] = 'comment'
post_centric_comments['content'] = post_centric_comments['body'].fillna('')

# Strip the Reddit prefixes to create purely alphanumeric, mappable IDs
# t1_ = reply to a comment | t3_ = reply directly to the post
post_centric_comments['clean_parent_id'] = post_centric_comments['parent_id'].str.replace('t1_', '').str.replace('t3_', '')
post_centric_comments['clean_root_id'] = post_centric_comments['link_id'].str.replace('t3_', '')


# ==========================================
# 3. Concatenate into a Unified Stream
# ==========================================
# Define the standardized schema for your social dynamics comparison
cols_to_keep = [
    'interaction_type',
    'subreddit',
    'id',
    'clean_parent_id',
    'clean_root_id',
    'author',
    'created_utc_dt',
    'content',
    'score'
]

# Safely extract matching columns
sub_cols = [c for c in cols_to_keep if c in sampled_submissions.columns]
com_cols = [c for c in cols_to_keep if c in post_centric_comments.columns]

unified_threads = pd.concat([
    sampled_submissions[sub_cols],
    post_centric_comments[com_cols]
], ignore_index=True)

# Sort by the Root Post ID, and then chronologically.
# This perfectly groups each thread together in the exact order the conversation happened.
unified_threads = unified_threads.sort_values(by=['clean_root_id', 'created_utc_dt']).reset_index(drop=True)
unified_threads["label"] = 1
print("\n=== FINAL UNIFIED THREAD STREAM ===")
print(unified_threads[['interaction_type', 'id', 'clean_parent_id', 'author']].head(10))
print(f"\nFinal Dataset Shape: {unified_threads.shape}")


=== FINAL UNIFIED THREAD STREAM ===
  interaction_type       id clean_parent_id                author
0       submission   100cxy            None             thesacred
1       submission   100y90            None             [deleted]
2          comment  j2kvtfk         100zfxn          Ill_Sound621
3          comment  j2kw3ny         100zfxn          catnapspirit
4          comment  j2kwgwj         100zfxn              coyote-1
5          comment  j2kyzc4         100zfxn         Oninonenbutsu
6          comment  j2l0r2f         100zfxn       Hanzo_The_Ninja
7          comment  j2l4cwd         j2kwgwj         Oninonenbutsu
8          comment  j2l6azr         100zfxn        Bakuretsu-Sama
9          comment  j2l90az         100zfxn  TheOverExcitedDragon

Final Dataset Shape: (8120, 10)


In [12]:
unified_threads.to_csv(f'{DESTINATION_DIR}/unified_threads.csv', index=False)

In [14]:
unified_threads.to_csv(f'unified_threads.csv', index=False)

In [ ]:
unified_stream.tail()